In [10]:
import os, requests, zipfile, gzip, io, shutil

def project_root():
    d = os.getcwd()
    for _ in range(8):
        if os.path.basename(d) in {"182", "183", "184"}:
            for sub in ("data", "DATA"):
                if os.path.isdir(os.path.join(d, sub)):
                    return d
        p = os.path.dirname(d)
        if p == d:
            break
        d = p
    return os.getcwd()

ROOT = project_root()
DATA_SUB = "DATA" if os.path.isdir(os.path.join(ROOT, "DATA")) else "data"
DATA_DIR = os.path.join(ROOT, DATA_SUB, "external")
os.makedirs(DATA_DIR, exist_ok=True)
print("data dir:", DATA_DIR)


data dir: c:\Users\ashut\Downloads\CASHNET\184\DATA\external


In [11]:
def download(url, name, timeout=180):
    dest = os.path.join(DATA_DIR, name)
    if os.path.exists(dest):
        print("skip", name, os.path.getsize(dest))
        return dest
    r = requests.get(url, stream=True, timeout=timeout)
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    print("saved", name, os.path.getsize(dest))
    return dest

def _load_kaggle_creds():
    import json as _json
    candidates = []
    if os.environ.get("KAGGLE_CONFIG_DIR"):
        candidates.append(os.environ["KAGGLE_CONFIG_DIR"])
    candidates.append(os.path.join(os.path.expanduser("~"), ".kaggle"))
    candidates.append(os.path.join(ROOT))
    candidates.append(os.path.join(os.path.expanduser("~"), "Downloads"))
    candidates.append(r"C:\Users\ashut\Downloads")
    for c in candidates:
        if not c:
            continue
        for fn in ("kaggle.json", "Kaggle.json"):
            p = os.path.join(c, fn)
            if os.path.exists(p):
                try:
                    d = _json.load(open(p))
                except Exception:
                    continue
                if d.get("key"):
                    os.environ["KAGGLE_API_TOKEN"] = d["key"]
                    os.environ["KAGGLE_KEY"] = d["key"]
                if d.get("username"):
                    os.environ["KAGGLE_USERNAME"] = d["username"]
                return p
    return None

def kaggle_fetch(slug, folder):
    try:
        import kaggle
    except Exception as e:
        print("kaggle package unavailable, skipping", slug, "->", e)
        return None
    _load_kaggle_creds()
    try:
        kaggle.api.authenticate()
    except Exception as e:
        print("kaggle auth failed, skipping", slug, "->", e)
        return None
    dest = os.path.join(DATA_DIR, folder)
    os.makedirs(dest, exist_ok=True)
    try:
        kaggle.api.dataset_download_files(slug, path=dest, unzip=True, quiet=False)
    except Exception as e:
        print("kaggle download failed, skipping", slug, "->", e)
        return None
    print("kaggle fetched", slug, "->", dest)
    return dest


In [12]:
import time, requests

def overpass_get(query, endpoints=None, max_tries=8):
    endpoints = endpoints or [
        "https://overpass-api.de/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",
        "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
    ]
    headers = {"User-Agent": "CASHNET-data-fetch/1.0", "Accept": "application/json"}
    wait = 10
    for attempt in range(max_tries):
        ep = endpoints[attempt % len(endpoints)]
        try:
            r = requests.post(ep, data={"data": query}, headers=headers, timeout=180)
            if r.status_code in (429, 504, 502, 500):
                print("endpoint busy", ep, "code", r.status_code, "sleep", wait)
                time.sleep(wait)
                wait = min(wait * 2, 120)
                continue
            r.raise_for_status()
            return r
        except requests.HTTPError as e:
            print("http error", e, "sleep", wait)
            time.sleep(wait)
            wait = min(wait * 2, 120)
    raise RuntimeError("overpass failed after retries")

cities = ["Delhi", "Mumbai", "Bengaluru", "Hyderabad", "Ahmedabad", "Gurugram"]
for city in cities:
    q = '[out:json][timeout:90];area["name"="{c}"]["admin_level"~"4|6"]->.a;(node["amenity"="atm"](area.a);way["amenity"="atm"](area.a););out center;'.format(c=city)
    r = overpass_get(q)
    out = os.path.join(DATA_DIR, "atm_" + city.lower() + ".json")
    with open(out, "w", encoding="utf-8") as f:
        f.write(r.text)
    print("saved", out, len(r.text))
    time.sleep(10)


saved c:\Users\ashut\Downloads\CASHNET\184\DATA\external\atm_delhi.json 245363
saved c:\Users\ashut\Downloads\CASHNET\184\DATA\external\atm_mumbai.json 339
saved c:\Users\ashut\Downloads\CASHNET\184\DATA\external\atm_bengaluru.json 339
saved c:\Users\ashut\Downloads\CASHNET\184\DATA\external\atm_hyderabad.json 339
endpoint busy https://overpass-api.de/api/interpreter code 429 sleep 10
endpoint busy https://overpass.kumi.systems/api/interpreter code 500 sleep 20
saved c:\Users\ashut\Downloads\CASHNET\184\DATA\external\atm_ahmedabad.json 338
saved c:\Users\ashut\Downloads\CASHNET\184\DATA\external\atm_gurugram.json 339


In [13]:
download("https://files.consumerfinance.gov/ccdb/complaints.csv.zip", "cfpb_complaints.csv.zip")


skip cfpb_complaints.csv.zip 1422909005


'c:\\Users\\ashut\\Downloads\\CASHNET\\184\\DATA\\external\\cfpb_complaints.csv.zip'

In [17]:
# Cell 1: Set Kaggle credentials manually
import os
import json
from pathlib import Path

# Option A: Set credentials directly (replace with your actual values)
KAGGLE_USERNAME = "ashutoshpatra3021"  # Replace with your Kaggle username
KAGGLE_API_KEY = "KGAT_df17acf7e3d474a278cafeb09498dc70"    # Replace with your Kaggle API key

# Create .kaggle directory if it doesn't exist
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

# Create kaggle.json file
kaggle_json_path = kaggle_dir / 'kaggle.json'
kaggle_data = {
    "username": KAGGLE_USERNAME,
    "key": KAGGLE_API_KEY
}

with open(kaggle_json_path, 'w') as f:
    json.dump(kaggle_data, f)

# Set proper permissions (Windows: this is optional but recommended)
try:
    os.chmod(kaggle_json_path, 0o600)
except:
    pass  # Permission setting may not work on Windows

print(f"✅ Kaggle credentials saved to: {kaggle_json_path}")
print(f"Username: {KAGGLE_USERNAME}")
print("Ready to download datasets!")

✅ Kaggle credentials saved to: C:\Users\ashut\.kaggle\kaggle.json
Username: ashutoshpatra3021
Ready to download datasets!


In [22]:
print("external data ready in", DATA_DIR)
for f in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, f)
    print(f, "dir" if os.path.isdir(p) else os.path.getsize(p))


external data ready in c:\Users\ashut\Downloads\CASHNET\184\DATA\external
atm_ahmedabad.json 352
atm_bengaluru.json 353
atm_delhi.json 259514
atm_gurugram.json 353
atm_hyderabad.json 353
atm_mumbai.json 353
cfpb_complaints.csv.zip 1422909005
synthetic_financial_fraud dir
